# Session 1 — GenAI & Agentic AI Architecture and Data

**Exercise: design a Security Architect agent** by mapping data source → retrieval / RAG → model → agent → memory → identity → permissions → human approval.

You will do it against three live systems that give a model knowledge in three different ways:

| Track | Method | Where the knowledge lives |
|---|---|---|
| Finance | fine-tuned Qwen2.5-3B (QLoRA adapter) | the adapter weights |
| Employee | 3.2M-parameter model trained **from scratch** | the model's weights |
| HR | RAG over ten HR documents | a vector index, read at inference |

Everything here runs with **your own Entra identity** — there are no API keys in this workshop.

## 0. Setup
Local: run `az login` in a terminal first. Colab: the next cell installs everything and the sign-in prints a device code.

In [ ]:
# Google Colab only: install the SDKs and fetch the workshop helpers. Local Jupyter/VS Code: skip.
import sys, subprocess, pathlib
if "google.colab" in sys.modules and not pathlib.Path("workshop.py").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "azure-ai-projects>=2", "azure-ai-agents>=1.1", "azure-identity>=1.17"], check=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/Auxin-io/Azure-GenAI-Security-Workshop.git", "_ws"], check=True)
    subprocess.run("cp -r _ws/workshop.py _ws/data . ", shell=True, check=True)
    print("Colab setup done - a device-code sign-in prompt will appear in the next cell")

In [ ]:
import workshop as w
print("signed in as", w.whoami())
client = w.agents_client()

## 1. Knowledge in the weights — the finance endpoint

The endpoint serves the same container twice: `use_adapter=false` is the untouched base model, `use_adapter=true` adds the LoRA adapter trained on the ten finance documents. **No document is sent with the question.**

In [ ]:
q = "How much do we owe Xenon Energy?"
for label, flag in (("BASE ", False), ("TUNED", True)):
    r = w.score("finance", q, use_adapter=flag, max_new_tokens=96)
    print(f"{label}: {r['answer']}   [{r['latency_ms']} ms]")

**Exercise 1.1** — ask three more questions and record which are right. Vendors in the weights: Yarrow Agriculture, Meridian Foods, Northwind Labs, Xenon Energy, Vantage Aerospace (invoices); Ironwood Supply, Zephyr Networks, Halcyon Print, Nordic Optics, Lakeshore Cabling (purchase orders). Include one vendor that is **not** in the list and note what happens.

In [ ]:
my_questions = [
    "When is the Meridian Foods invoice due?",
    "What is the Zephyr Networks purchase order number?",
    "What is the Cedar Systems invoice total?",     # not in the ten
]
for q in my_questions:
    print(q, "->", w.score("finance", q)["answer"])

## 2. A model with *only* this knowledge — the employee endpoint

3.2M parameters, random initialisation, trained for 77 seconds on 330 question/answer rows. It knows nothing except its ten timesheets and expense reports.

In [ ]:
for q in ["How many hours did Jonas Weber work?",
          "What is the status of Aisha Rahman's expense report?",
          "What is the capital of France?"]:
    print(q, "->", w.score("employee", q)["answer"])

**Exercise 1.2** — the last answer is nonsense. Write one sentence on *why* a from-scratch model behaves this way and what the finance track has that this one does not.

_Your answer:_ 

## 3. Knowledge in an index — the HR RAG agent

Nothing was trained. Ten OCR'd HR texts were uploaded to a Foundry vector store (chunk → embed → index); the agent retrieves matching chunks at question time and cites the file.

In [ ]:
hr = w.find_agent(client, w.CONFIG["agents"]["hr"])
t = w.ask(client, hr.id, "How much notice does the Flexible Hours Policy require?")
w.show(t)
print()
w.describe_steps(t)     # the agent loop: retrieval step, then the message

In [ ]:
# the same agent refuses when nothing is retrievable
w.show(w.ask(client, hr.id, "What is the parental leave allowance?"))

## 4. The agent in front of the weights — the finance agent

`gpt-4.1-mini` does no finance reasoning of its own. It decides *whether* to call the tool, calls the endpoint with the project's **managed identity**, and relays the answer verbatim.

In [ ]:
fin = w.find_agent(client, w.CONFIG["agents"]["finance"])
for q in ["How much do we owe Xenon Energy?", "What is the capital of France?"]:
    t = w.ask(client, fin.id, q)
    w.show(t)
print()
print("tools on the agent:", [tool["type"] for tool in fin.tools])

## 5. Build the agent — all three knowledge sources, one agent

Sections 1–4 used agents that already existed. Now you create one, and it is **yours**: it carries your alias, and you delete it at the end.

The point is that the three tracks are not three architectures. They are three **tools on one agent**, and the model in front decides which to reach for. Knowledge in adapter weights, knowledge in from-scratch weights, and knowledge in an index all arrive at the agent the same way.

In [ ]:
# Your Entra sign-in name, trimmed - so thirty attendees do not collide on one agent name.
ALIAS = w.sample_alias()
AGENT_NAME = f"architect-agent-{ALIAS}"
print("your agent will be called", AGENT_NAME)

In [ ]:
from azure.ai.agents.models import (FileSearchTool, OpenApiTool,
                                    OpenApiManagedAuthDetails, OpenApiManagedSecurityScheme)

# Tools 1 and 2: the two Azure ML endpoints, described to the agent as OpenAPI operations.
# The agent calls them with the PROJECT's managed identity - no key ever enters this notebook.
ml_auth = OpenApiManagedAuthDetails(
    security_scheme=OpenApiManagedSecurityScheme(audience="https://ml.azure.com"))

finance_tool = OpenApiTool(
    name="finance_model",
    description="Answers questions about the ten finance documents (invoices, purchase orders, "
                "vendors, amounts, due dates) from a fine-tuned model. No document is supplied.",
    spec=w.openapi_spec("finance"), auth=ml_auth)

employee_tool = OpenApiTool(
    name="employee_model",
    description="Answers questions about employee timesheets and expense reports from a model "
                "trained from scratch on those ten documents only.",
    spec=w.openapi_spec("employee"), auth=ml_auth)

# Tool 3: retrieval. Your own vector store, built from the HR text files in data/hr.
store = w.build_vector_store(client, f"hr-store-{ALIAS}")
hr_tool = FileSearchTool(vector_store_ids=[store.id])
print("vector store", store.id, "-", store.file_counts.completed, "files indexed")

In [ ]:
INSTRUCTIONS = """You are an internal documents assistant with three tools.

Finance questions (invoices, purchase orders, vendors, amounts, due dates): call finance_model with
the user's question unchanged and reply with its answer verbatim.
Employee questions (timesheets, hours worked, expense reports): call employee_model the same way.
HR questions (policies, leave): use file search, answer only from the retrieved text, and cite the
source file name.

Never answer from general knowledge and never invent a number. If a tool says it does not have the
document, say exactly that."""

tools = finance_tool.definitions + employee_tool.definitions + hr_tool.definitions
existing = next((a for a in client.list_agents() if a.name == AGENT_NAME), None)
if existing:
    agent = client.update_agent(existing.id, model=w.CONFIG["model"], instructions=INSTRUCTIONS,
                                tools=tools, tool_resources=hr_tool.resources)
else:
    agent = client.create_agent(model=w.CONFIG["model"], name=AGENT_NAME,
                                instructions=INSTRUCTIONS, tools=tools,
                                tool_resources=hr_tool.resources)
print("agent", agent.id)
print("tools:", [t["type"] if isinstance(t, dict) else t.type for t in agent.tools])

### One question per track

Watch **which tool fires**. `describe_steps` prints the agent loop: the tool call, its arguments, then the message. An answer with *no* tool call is the failure to look for — the model answered from its own knowledge instead of your documents.

In [ ]:
for q in ["How much do we owe Xenon Energy?",                        # -> finance_model
          "How many hours did Jonas Weber work?",                    # -> employee_model
          "How much notice does the Flexible Hours Policy require?", # -> file search
          "What is the capital of France?"]:                         # -> no tool; answers normally
    t = w.ask(client, agent.id, q)
    w.show(t)
    w.describe_steps(t)
    print()

**Before you move on.** Your agent reached three different knowledge stores using **one** identity — the project's managed identity. What would you change so that a user allowed to read HR policies but *not* finance documents has that enforced? Write down where in the picture the check has to live. Session 4 returns to exactly this.

In [ ]:
# Clean up what you created. Leave this to the end of the session.
client.delete_agent(agent.id)
client.vector_stores.delete(store.id)
print("deleted", AGENT_NAME, "and its vector store")

## 6. Exercise — map the Security Architect agent

Fill in the dictionary below for **one** of the three tracks. Use the resource names you saw above. Then answer the two questions.

In [ ]:
design = {
    "track":            "finance | employee | hr",
    "data_source":      "",     # where the documents come from
    "preparation":      "",     # OCR? chunking? labelled rows?
    "knowledge_store":  "",     # weights / adapter / vector store
    "model":            "",     # which model answers, which model reasons
    "agent":            "",     # agent name and its single tool
    "memory":           "",     # what holds conversation state
    "identity":         "",     # who calls the endpoint, with what token
    "permissions":      "",     # which role, on which resource
    "human_approval":   "",     # where would you put one, and for what action
}
for k, v in design.items():
    print(f"{k:<18} {v}")

1. Which of the three tracks would you choose for data that changes every week — and why?
2. Which track leaks the most if the model file is stolen — and why?

_Your answers:_